In [1]:
import polars as pl
import duckdb
import os
from pathlib import Path
from typing import Tuple, List, Dict, Union
import seaborn as sns
import plotly.express as px

In [2]:
DATA_DIR=Path("../../Sepsis-data/data/raw_data_phase2_v2")
os.listdir(DATA_DIR)

['Flowsheets - Mar 2025 - Feb 2026 - 4.9.26.csv',
 'Lab Results - Mar 2025 - Feb 2026 - 4.8.26.csv',
 'Encounters - Mar 2025 - Feb 2026 - 3.31.26.csv',
 'Lab Results - 3.9.26.csv',
 'Flowsheet - 3.9.26.csv',
 'Med Admin - Mar 2025 - Feb 2026 - 4.8.26.csv',
 'Procedure Orders - 3.9.26.csv',
 'Flowsheets - Mar 2025 - Feb 2026 - 4.9.26.csv.zip',
 'Encounters with Scores and Flags - Mar 2025 - Feb 2026 - 3.31.26.csv',
 'Diagnoses - 3.9.26.csv',
 'Procedure Orders - Mar 2025 - Feb 2026 - 4.9.26.csv',
 'Flowsheet SBP - 3.9.26.csv',
 'Encounter Table with Baseline Values - Mar 2025 - Feb 2026 - 3.25.26.csv',
 'Med Admin - 3.9.26.csv',
 'Scores and Flags - Mar 2025 - Feb 2026 - 3.31.26.csv',
 'Flowsheet Events Feb 2025 - Mar 2026 - 3.31.26.csv',
 'Encounter Table with Baseline Values - Mar 2025 - Feb 2026.csv',
 'Flowsheet - 3.24.26.csv',
 'Encounter Table with Baseline Values - Mar 2025 - Feb 2026 - 4.2.26.csv']

In [3]:
df_enc = pl.read_csv(DATA_DIR/Path("Encounter Table with Baseline Values - Mar 2025 - Feb 2026 - 4.2.26.csv"))

In [4]:
df_enc.columns

['PrimaryMrn',
 'EncounterEpicCsn',
 'AdmissionDateValue',
 'DischargeDateValue',
 'Arrival_Instant',
 'FirstAdmissionOrderInstant',
 'InpatientAdmissionInstant',
 'Admitted_from_ED',
 'PatientClass',
 'InpatientAdmissionPatientClass',
 'HospitalService',
 'AdmittingDepartment',
 'DischargeDepartment',
 'AdmissionType',
 'AdmissionSource',
 'AdmissionOrigin',
 'PrincipalProblem',
 'PrimaryCodedDiagnosis',
 'PrimaryCodedProcedureKey',
 'Death_Flag',
 'Original_POA_Condition',
 'Sepsis_Category',
 'Baseline_SBP',
 'Baseline_RespiratoryRate',
 'Baseline_PulseRate',
 'Baseline_Creatinine',
 'Baseline_Platelets',
 'Baseline_Bilirubin',
 'Baseline_eGFR',
 'Baseline_WBC',
 'Suspected_Infection_Time',
 'Suspected_Infection_Criteria_Met',
 'Cancer_Registry_YN',
 'HIV_Registry_YN',
 'Immunocrompromised_Registry_YN',
 'CKD_Dialysis_Registry_YN',
 'Solid_Organ_Transplant_Registry_YN',
 'Pregnancy_Registry_YN']

In [4]:
# lab_res = pl.read_csv(DATA_DIR/Path("Lab Results - 3.9.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])
# flowsheets = pl.read_csv(DATA_DIR/Path("Flowsheet - 3.24.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])
# # flowsheets_sbp = pl.read_csv(DATA_DIR/Path("Flowsheet SBP - 3.9.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])
# med_admin = pl.read_csv(DATA_DIR/Path("Med Admin - 3.9.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])
# procedures = pl.read_csv(DATA_DIR/Path("Procedure Orders - 3.9.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])
# baseline = pl.read_csv(DATA_DIR/Path("Encounter Table with Baseline Values - Mar 2025 - Feb 2026 - 3.25.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])
# diagnosis = pl.read_csv(DATA_DIR/Path("Diagnoses - 3.9.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])

In [4]:
flowsheets_old = pl.read_csv(DATA_DIR/Path("Flowsheet - 3.24.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])
flowsheets_new = pl.read_csv(DATA_DIR/Path("Flowsheet Events Feb 2025 - Mar 2026 - 3.31.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])

In [5]:
flowsheets_old.shape, flowsheets_new.shape

((6643434, 8), (6643896, 8))

In [8]:
flowsheets_old.schema

Schema([('EncounterEpicCsn', String),
        ('Event_DateTime', String),
        ('Type', String),
        ('Event_Grouper', String),
        ('Event_Name', String),
        ('NumericValue', String),
        ('Value', String),
        ('Flag', String)])

In [9]:
flowsheets_new.schema

Schema([('EncounterEpicCsn', String),
        ('Event_DateTime', String),
        ('Type', String),
        ('Event_Grouper', String),
        ('Event_Name', String),
        ('NumericValue', String),
        ('Value', String),
        ('Flag', String)])

In [12]:
set(flowsheets_old['Event_Grouper'].unique()) - set(flowsheets_new['Event_Grouper'].unique()), set(flowsheets_new['Event_Grouper'].unique()) - set(flowsheets_old['Event_Grouper'].unique())

({'O2 Delivery High-Flow Nasal Cannula'}, {'O2 Delivery High-Flow'})

In [19]:
flowsheets_old.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow Nasal Cannula").select(pl.col("Value").unique())['Value'].to_list()

['high-flow mask', 'Venturi mask system', 'high-flow nasal cannula']

In [21]:
flowsheets_new.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow").select(pl.col("Value").unique())['Value'].to_list()

['humidified;high-flow nasal cannula',
 'high-flow mask',
 'high-flow nasal cannula',
 'Venturi mask system',
 'high-flow nasal cannula;heated',
 'high-flow nasal cannula;humidified']

In [17]:
flowsheets_old.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow Nasal Cannula").shape, flowsheets_new.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow").shape

((5730, 8), (17772, 8))

In [23]:
flowsheets_old.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow Nasal Cannula")['Event_Name'].value_counts()

Event_Name,count
str,u64
"""UTSW R ED PRE HOSPITAL CARE EM…",2
"""CPM S25 R INV DEVICE.INV O2 DE…",5728


In [22]:
flowsheets_new.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow")['Event_Name'].value_counts()

Event_Name,count
str,u64
"""CPM S25 R INV DEVICE.INV O2 DE…",17770
"""UTSW R ED PRE HOSPITAL CARE EM…",2


In [30]:
fig = px.histogram(flowsheets_old.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow Nasal Cannula").to_pandas(), x="Value", title="Flowsheet Old - O2 Delivery High-Flow Nasal Cannula")
fig.update_xaxes(categoryorder='total descending')

In [29]:
fig = px.histogram(flowsheets_new.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow").to_pandas(), x="Value", title="Flowsheet New - O2 Delivery High-Flow",)
fig.update_xaxes(categoryorder='total descending')

---------

In [3]:
!ls ../../Sepsis-data/data/output/raw_data_phase2_v2/

df_agg.parquet	     df_all.parquet	df_organdysfunction.parquet
df_agg_vent.parquet  df_infect.parquet	df_septicshock.parquet


In [6]:
df_agg = pl.read_parquet(Path("../../Sepsis-data/data/output/raw_data_phase2_v2/df_agg.parquet"))

In [9]:
flowsheets_new = pl.read_csv(DATA_DIR/Path("Flowsheet Events Feb 2025 - Mar 2026 - 3.31.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])

In [ ]:
flowsheets_new = flowsheets_new.with_columns(
    pl.col("Event_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f"),
	pl.col("NumericValue").cast(pl.Float64),
    pl.col("EncounterEpicCsn").cast(pl.Int64)
)
flowsheets_new = flowsheets_new.sort(by=["EncounterEpicCsn", "Event_DateTime"])

In [26]:
flowsheets_new.filter(pl.col("Value") == "Venturi mask system")

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag
i64,datetime[μs],str,str,str,f64,str,str
699476058,2024-07-11 11:35:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""Venturi mask system""",null
699476058,2025-05-06 15:35:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""Venturi mask system""",null
714534044,2024-11-08 19:38:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""Venturi mask system""",null
714534044,2024-12-26 03:00:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""Venturi mask system""",null
716208986,2025-02-08 10:38:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""Venturi mask system""",null
…,…,…,…,…,…,…,…
745516802,2026-02-02 21:48:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""Venturi mask system""",null
745516802,2026-02-02 22:59:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""Venturi mask system""",null
745516802,2026-02-03 14:36:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""Venturi mask system""",null


In [24]:
flowsheets_new.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow")

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag
i64,datetime[μs],str,str,str,f64,str,str
699476058,2024-07-11 11:35:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""Venturi mask system""",null
699476058,2024-07-26 11:00:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""humidified;high-flow nasal can…",null
699476058,2024-08-05 05:51:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null
699476058,2024-12-25 14:34:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null
699476058,2024-12-27 10:15:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null
…,…,…,…,…,…,…,…
746919560,2026-02-21 11:00:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null
746919560,2026-02-21 11:10:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula;humidi…",null
746919560,2026-02-21 12:02:00,"""Flowsheet""","""O2 Delivery High-Flow""","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""high-flow nasal cannula""",null


In [27]:
flowsheets_new.filter(pl.col("Event_Grouper") == "O2 Delivery High-Flow").group_by("EncounterEpicCsn").agg(pl.col("Event_DateTime").min())

EncounterEpicCsn,Event_DateTime
i64,datetime[μs]
699476058,2024-07-11 11:35:00
709516764,2025-01-31 21:00:00
714445951,2024-10-24 16:28:00
714534044,2024-10-26 09:47:00
716208986,2025-02-08 10:25:00
…,…
746580472,2026-02-24 09:08:00
746672537,2026-02-22 09:00:00
746825096,2026-02-19 20:11:00


# ================================

In [8]:
encounter_old = pl.read_csv(DATA_DIR/Path("Encounter Table with Baseline Values - Mar 2025 - Feb 2026 - 3.25.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])
# encounter_new = pl.read_csv(DATA_DIR/Path("Encounters with Scores and Flags - Mar 2025 - Feb 2026 - 3.31.26.csv"), infer_schema=False, null_values=['Null', "NULL", 'null'])

In [4]:
encounter_old.shape, encounter_new.shape

((3979, 31), (1929174, 90))

In [26]:
(1929174/3979)*(90/31)

1407.596818782479

In [10]:
old_group = encounter_old.group_by("EncounterEpicCsn").agg(
    [pl.col(c).last() for c in encounter_old.columns if c.startswith("Baseline")]+
    [pl.col(c).last() for c in encounter_old.columns if 'Date' in c or 'Instant' in c]
)

In [6]:
new_group = encounter_new.group_by("EncounterEpicCsn").agg(
    [pl.col(c).last() for c in encounter_new.columns if c.startswith("Baseline")]+
    [pl.col(c).last() for c in encounter_new.columns if 'Date' in c or 'Instant' in c]
)


In [ ]:
jnd = old_group.join(new_group, on="EncounterEpicCsn", how="left")

In [22]:
for col in old_group.columns:
	if col.startswith("Baseline"):
		print(col, jnd.filter(
			(pl.col(col) != pl.col(col+"_right"))&
			(pl.col("EncounterEpicCsn") != '746919986') # replace with actual CSN to inspect
			).shape[0])
# jnd.with_columns(
#     pl.when(pl.col("Baseline_SBP") != pl.col("Baseline_SBP_right")).then(1).otherwise(0).alias("Discrepancy_SBP")
# )['Discrepancy_SBP'].sum()

Baseline_SBP 0
Baseline_RespiratoryRate 0
Baseline_PulseRate 0
Baseline_Creatinine 100
Baseline_Platelets 0
Baseline_Bilirubin 63
Baseline_eGFR 0
Baseline_WBC 125


In [24]:
jnd.filter(
    pl.col("Baseline_Creatinine") != pl.col("Baseline_Creatinine_right")
).select("EncounterEpicCsn", "Baseline_Creatinine", "Baseline_Creatinine_right").to_pandas()

,EncounterEpicCsn,Baseline_Creatinine,Baseline_Creatinine_right
0,728129125,0.625652173913043,0.625652173913044
1,720487026,0.820714285714285,0.820714285714286
2,742324778,0.423260869565218,0.423260869565217
3,735409476,0.848636363636364,0.848636363636363
4,746886090,11.9631481481481,11.9631481481482
...,...,...,...
95,743778991,0.473646616541354,0.473646616541353
96,720110308,0.788571428571428,0.788571428571429
97,732648029,0.685454545454546,0.685454545454545
98,720117699,3.17523255813954,3.17523255813953


In [11]:
old_group.shape, new_group.shape

((3979, 13), (3980, 15))

In [12]:
old_group.schema, new_group.schema

(Schema([('EncounterEpicCsn', String),
         ('Baseline_SBP', String),
         ('Baseline_RespiratoryRate', String),
         ('Baseline_PulseRate', String),
         ('Baseline_Creatinine', String),
         ('Baseline_Platelets', String),
         ('Baseline_Bilirubin', String),
         ('Baseline_eGFR', String),
         ('Baseline_WBC', String),
         ('AdmissionDateValue', String),
         ('DischargeDateValue', String),
         ('Arrival_Instant', String),
         ('InpatientAdmissionInstant', String)]),
 Schema([('EncounterEpicCsn', String),
         ('Baseline_SBP', String),
         ('Baseline_RespiratoryRate', String),
         ('Baseline_PulseRate', String),
         ('Baseline_Creatinine', String),
         ('Baseline_Platelets', String),
         ('Baseline_Bilirubin', String),
         ('Baseline_eGFR', String),
         ('Baseline_WBC', String),
         ('AdmissionDateValue', String),
         ('DischargeDateValue', String),
         ('Arrival_Instant', Strin

In [ ]:
jset(new_group['EncounterEpicCsn']) - set(old_group['EncounterEpicCsn'])


{'746919986'}